# Face Recognition Project

## 1. Introduction

## 2. Get Setup

### 2.1 Download Dataset Helper Function

In [1]:
import os
import zipfile
from pathlib import Path
import requests

def download_data(source: str, 
                    destination: str,
                    remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
        destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        print(f"[INFO] Downloading {target_file} from {source}...")
        with requests.get(source, stream=True, timeout=30) as response:
            response.raise_for_status()
            with open(data_path / target_file, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

### 2.2 Unzip Dataset

In [2]:
from pathlib import Path
import zipfile

dataset_dir = Path("dataset")
zip_path = Path("data/dataset.zip")

if dataset_dir.exists():
    print("[INFO] Dataset already available.")
elif zip_path.exists():
    print("[INFO] Unzipping dataset...")
    dataset_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dataset_dir)
else:
    print("[INFO] Dataset zip not found locally... downloading dataset...")
    download_data(source="https://github.com/Axeloooo/Face-Recognition/raw/devel/data/dataset.zip",
                    destination="dataset")

[INFO] Dataset zip not found locally... downloading dataset...
[INFO] data/dataset directory exists, skipping download.


## 3. Get Data

In [3]:
import cv2
import numpy as np
from glob import glob
from pathlib import Path

# custom load data function to load images and record subject labels
def get_data(path: str):
    paths = glob(path, recursive=True)
    data = [] #list of images
    label = [] #list of labels

    for path in paths:
        img = cv2.imread(path,0) # read image
        # Extract subject label from path using pathlib (OS-independent)
        path_parts = Path(path).parts
        subject_folder = [part for part in path_parts if part.startswith('s') and part[1:].isdigit()]
        if subject_folder:
            subject_label = subject_folder[0][1:]  # Remove 's' prefix
        else:
            continue  # Skip if subject folder not found

        # pre−processing step
        # can resize, rescale, normalize
        img = img.reshape(-1) # reshape image to a 1D vector
        img = np.float32(img / 255.0) #normalize to 0−1 value

        # can apply LBP, PCA or other forms of feature extraction
        # append images and labels
        data.append(img)

        # decrease all labels by 1 since subject labels start from 1
        label.append(int(subject_label)-1)
        
    return np.array(data), np.array(label)

## 4. Local Binary Pattern (LBP) 

In [4]:
from skimage.feature import local_binary_pattern
from skimage.io import imread
import numpy as np

# Read one image of subject 1 from dataset
img = imread("data/dataset/ATT dataset/s1/1.pgm", as_gray=True)

# Extract LBP features from the image
# P: Number of circularly symmetric neighbor set points = 12
# R: Radius of circle = 3
# method='default' produces raw decimal LBP codes in the range [0, 2^P - 1]
lbp = local_binary_pattern(img, P=12, R=3, method='default')

# The LBP output is an image of the same shape as the input; each pixel value
# encodes the local texture pattern at that location.  To use it as a feature
# vector we build a histogram over all pixel values.
# For P=12, LBP values span [0, 2^12 - 1] = [0, 4095].
n_bins = 64          # number of histogram bins — tuned in the experiment below
lbp_max = 2 ** 12   # upper bound of value range for P=12

hist, bin_edges = np.histogram(lbp.ravel(), bins=n_bins, range=(0, lbp_max), density=True)

print(f"LBP image shape : {lbp.shape}   (same as input)")
print(f"Histogram shape : {hist.shape}  (feature vector length = {n_bins})")
print(f"Histogram sum   : {hist.sum() * (bin_edges[1] - bin_edges[0]):.4f}  (≈ 1.0 for density=True)")

LBP image shape : (112, 92)   (same as input)
Histogram shape : (64,)  (feature vector length = 64)
Histogram sum   : 1.0000  (≈ 1.0 for density=True)


In [5]:
from skimage.feature import local_binary_pattern
import cv2
import numpy as np
from glob import glob
from pathlib import Path

def get_data_lbp(path: str, n_bins: int = 64):
    """Load images and extract LBP histogram feature vectors.

    Each image is converted to a normalised histogram of Local Binary Pattern
    values (P=12 neighbours, radius R=3, method='default').  The histogram
    acts as a compact, fixed-length texture descriptor of length n_bins.

    Args:
        path (str): Glob pattern pointing to .pgm image files.
        n_bins (int): Number of histogram bins (controls descriptor length).
                      For P=12, LBP values span [0, 2^12 - 1] = [0, 4095].

    Returns:
        Tuple[np.ndarray, np.ndarray]: (data, labels) arrays.
    """
    paths = glob(path, recursive=True)
    data, label = [], []
    lbp_max = 2 ** 12  # value range upper bound for P=12

    for p in paths:
        img = cv2.imread(p, 0)
        if img is None:
            continue
        path_parts = Path(p).parts
        subject_folder = [part for part in path_parts if part.startswith('s') and part[1:].isdigit()]
        if not subject_folder:
            continue

        img_f = np.float32(img / 255.0)
        lbp = local_binary_pattern(img_f, P=12, R=3, method='default')
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, lbp_max), density=True)

        data.append(hist)
        label.append(int(subject_folder[0][1:]) - 1)

    return np.array(data), np.array(label)

In [6]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import numpy as np

_train_path = "data/dataset/ATT dataset/s*/[1-8].pgm"

# --- Baseline: raw pixel intensities ---
_raw_data, _raw_label = get_data(_train_path)
_baseline_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(C=1.0, gamma='scale', random_state=42)),
])
baseline_score = cross_val_score(
    _baseline_pipe, _raw_data, _raw_label, cv=5, scoring='accuracy', n_jobs=-1
).mean()
print(f"Raw pixels        CV accuracy: {baseline_score:.4f}")

# --- LBP with varying histogram bin counts ---
# A simple fixed SVC (C=1, gamma='scale') is used here so that differences in
# accuracy reflect the feature representation, not hyperparameter tuning.
bin_candidates = [16, 32, 64, 128, 256]
lbp_scores = {}
for n_bins in bin_candidates:
    _lbp_data, _lbp_label = get_data_lbp(_train_path, n_bins=n_bins)
    _lbp_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(C=1.0, gamma='scale', random_state=42)),
    ])
    score = cross_val_score(
        _lbp_pipe, _lbp_data, _lbp_label, cv=5, scoring='accuracy', n_jobs=-1
    ).mean()
    lbp_scores[n_bins] = score
    print(f"LBP n_bins={n_bins:>3d}    CV accuracy: {score:.4f}")

best_n_bins = max(lbp_scores, key=lbp_scores.get)
best_lbp_score = lbp_scores[best_n_bins]

print(f"\nBaseline (raw pixels)        : {baseline_score:.4f}")
print(f"Best LBP (n_bins={best_n_bins:>3d})     : {best_lbp_score:.4f}")

use_lbp = best_lbp_score > baseline_score
print(f"\nSection 5 will use: {'LBP histogram (n_bins=' + str(best_n_bins) + ')' if use_lbp else 'raw pixel intensities'}")

Raw pixels        CV accuracy: 0.9531


/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


LBP n_bins= 16    CV accuracy: 0.7000


/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


LBP n_bins= 32    CV accuracy: 0.7562


...

LBP n_bins= 64    CV accuracy: 0.8094


/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


LBP n_bins=128    CV accuracy: 0.8594


/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


LBP n_bins=256    CV accuracy: 0.8531

Baseline (raw pixels)        : 0.9531
Best LBP (n_bins=128)     : 0.8594

Section 5 will use: raw pixel intensities


## 5. Support Vector Machines (SVM)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/support-vector-machines.png)

In [7]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

_base = "data/dataset/ATT dataset/s*"
train_path = f"{_base}/[1-8].pgm"

# Select feature extraction method based on Section 4 experiment results
loader = get_data_lbp if use_lbp else get_data
loader_kwargs = {'n_bins': best_n_bins} if use_lbp else {}

train_data,   train_label   = loader(train_path,         **loader_kwargs)
test_data_9,  test_label_9  = loader(f"{_base}/9.pgm",  **loader_kwargs)
test_data_10, test_label_10 = loader(f"{_base}/10.pgm", **loader_kwargs)
test_data  = np.concatenate([test_data_9,  test_data_10])
test_label = np.concatenate([test_label_9, test_label_10])

if len(train_data) == 0 or len(train_label) == 0:
    raise ValueError(f"No training data found for pattern: {train_path}.")
if len(test_data) == 0 or len(test_label) == 0:
    raise ValueError("No test data found.")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42)),
])

param_grid = {
    'svc__kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'svc__C': [0.1, 1.0, 5.0, 10.0, 100.0],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1],
}

svm = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)
svm.fit(train_data, train_label)

print(f"Best parameters : {svm.best_params_}")
print(f"Best CV accuracy: {svm.best_score_:.4f}")

# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = svm.predict_proba(test_data)

# calculate accuracy
prediction = np.argmax(probability_matrix, 1)
result = prediction == test_label
accuracy = np.sum(result) / len(result)

print(f"Test accuracy   : {accuracy:.4f}")

Best parameters : {'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
Best CV accuracy: 0.9750
Test accuracy   : 0.9625


## 6. Multi-Layer Perceptron (MLP)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/multi-layer-perceptron.png)

In [8]:
from sklearn.neural_network import MLPClassifier
import numpy as np

# TODO: Update these glob patterns if your dataset is stored in a different location.
# Example split for datasets organized like dataset/s1/1.pgm ... dataset/s40/10.pgm
train_path = "dataset/s*/[1-8].pgm"
test_path = "dataset/s*/[9-10].pgm"

# customize get data function to include pre−processing methods (adding PCA or LBP)
train_data , train_label = get_data(train_path) # use the previous custom get data function
test_data , test_label = get_data(test_path) # use the previous custom get data function

if len(train_data) == 0 or len(train_label) == 0:
    raise ValueError(f"No training data found for pattern: {train_path}. Update train_path to match your dataset files.")
if len(test_data) == 0 or len(test_label) == 0:
    raise ValueError(f"No test data found for pattern: {test_path}. Update test_path to match your dataset files.")

# create MLP with 3 layers of perceptrons
# first layers has 128 neurons then 64 then another 128
# experiment with different layers/neurons
# experiment with different learning rate
mlp = MLPClassifier(hidden_layer_sizes=(128,64,128),
                    learning_rate_init=0.001,
                    random_state=1)
mlp.fit(train_data , train_label)

# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = mlp.predict_proba(test_data)

# calculate accuracy
prediction = np.argmax(probability_matrix ,1)
result = prediction == test_label
accuracy = np.sum(result)/len(result)

ValueError: No training data found for pattern: dataset/s*/[1-8].pgm. Update train_path to match your dataset files.

## 7. ROC (FPR vs. TPR)

## 8. DET (FPR vs. FNR)

## 9. Conclusion